# 3. Apply LLM Extraction To Filtered Corpus

This notebook handles the full-corpus LLM application step: building/maintaining the routing plan for `filtered_texts`, submitting batch/direct jobs, checking status, downloading outputs, and parsing run outputs into result CSVs.


In [ ]:
# Resolve the repository and external data directory without machine-specific paths.
from pathlib import Path
import sys

for _candidate in (Path.cwd(), *Path.cwd().parents):
    if (_candidate / "project_config.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError("Run this notebook from within the cloned repository.")

from project_config import REPO_ROOT, DATA_ROOT, require_env

import csv
import json
import math
import os
import re
import time
import urllib.error
import urllib.request
import uuid
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

# =========================================================
# SETTINGS
# =========================================================
from pathlib import Path

# Code is run from the OneDrive project code folder.
CODE_ROOT = REPO_ROOT
TEXT_MINING_CODE_ROOT = CODE_ROOT / "text_mining"

# Data is read from and written to the share-drive project folder.
SHARE_ROOT = DATA_ROOT
# DATA_ROOT is configured through project_config.
ROOT = DATA_ROOT
INPUT_FOLDER = ROOT / "filtered_texts"
OUTPUT_ROOT = ROOT / "llm_labelled_data"
ROUTING_DIR = OUTPUT_ROOT / "routing"
BATCH_RUNS_DIR = OUTPUT_ROOT / "batch_runs"
DIRECT_RUNS_DIR = OUTPUT_ROOT / "direct_runs"
PROMPT_FILE = TEXT_MINING_CODE_ROOT / "prompts" / "data_source_extraction_system_prompt.txt"

MODEL = "gpt-5.4-mini"
PROMPT_VARIANT = "guidelines_simple"
PROMPT_REVISION = "author_manual_toolfix_20260531"
SAFETY_IDENTIFIER = os.environ.get(
    "OPENAI_SAFETY_IDENTIFIER",
    "interdependence_data_research_corpus_labelling",
)

FILE_MANIFEST_FILE = ROUTING_DIR / "filtered_texts_file_manifest.csv"
BATCH_PLAN_FILE = ROUTING_DIR / "filtered_texts_batch_routing_plan.csv"

# Chunking defaults. With an assumed 40M queued-token limit, one ~4,500-file chunk
# is usually about 15-16M input tokens for this corpus.
TARGET_FILES_PER_BATCH = 4500
ACTIVE_TOKEN_CAP = 98_000_000
MAX_INPUT_TOKENS = 200_000
CHARS_PER_TOKEN_ESTIMATE = 4.0
INPUT_TOKEN_ESTIMATE_PER_BYTE = 0.33
EXPECTED_OUTPUT_TOKENS_PER_REQUEST = 65

# OpenAI safety identifier:
# - sent with every Responses API call, including Batch JSONL requests
# - should be stable for this end-user/workflow
# - should not contain direct personal information
#
# If you want a different identifier, set OPENAI_SAFETY_IDENTIFIER in your
# environment before running the notebook.

# Batch pricing estimates: 50% of the direct-run prices used earlier.
BATCH_PRICE_INPUT_PER_1M = 0.125
BATCH_PRICE_OUTPUT_PER_1M = 1.00
DIRECT_PRICE_INPUT_PER_1M = 0.25
DIRECT_PRICE_OUTPUT_PER_1M = 2.00

TEXT_FORMAT = {
    "type": "json_schema",
    "name": "guidelines_database_extraction",
    "schema": {
        "type": "object",
        "properties": {
            "db_use": {"type": "integer", "enum": [0, 1]},
            "data_sources": {"type": "array", "items": {"type": "string"}},
            "reason_short": {"type": "string"},
        },
        "required": ["db_use", "data_sources", "reason_short"],
        "additionalProperties": False,
    },
    "strict": True,
}

ROUTING_DIR.mkdir(parents=True, exist_ok=True)
BATCH_RUNS_DIR.mkdir(parents=True, exist_ok=True)
DIRECT_RUNS_DIR.mkdir(parents=True, exist_ok=True)


def require_api_key():
    api_key = os.environ.get("OPENAI_API_KEY")
    if not api_key:
        raise RuntimeError("OPENAI_API_KEY is not set")
    return api_key


SYSTEM_PROMPT = PROMPT_FILE.read_text(encoding="utf-8")


def normalize_whitespace(text):
    return re.sub(r"\s+", " ", str(text).replace("\x00", " ")).strip()


def read_txt_file(filepath):
    try:
        return filepath.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        return filepath.read_text(encoding="latin-1")


def prepare_text(text):
    text = normalize_whitespace(text)
    max_chars = int(MAX_INPUT_TOKENS * CHARS_PER_TOKEN_ESTIMATE)
    if len(text) <= max_chars:
        return text, False
    return text[:max_chars], True


def normalize_label_value(value, strip_chars=";,."):
    if value is None:
        return ""
    value = re.sub(r"\s+", " ", str(value)).strip().strip(strip_chars)
    if not value:
        return ""
    if value.lower() == "manual":
        return "MANUAL"
    return value


def deduplicate_sources(sources):
    cleaned = []
    seen = set()
    for value in sources or []:
        source = normalize_label_value(value)
        if not source:
            continue
        key = source.lower()
        if key in seen:
            continue
        seen.add(key)
        cleaned.append(source)
    return cleaned


def serialize_json_field(value):
    return json.dumps(value, ensure_ascii=False)


def estimate_cost(input_tokens, n_requests, mode):
    output_tokens = n_requests * EXPECTED_OUTPUT_TOKENS_PER_REQUEST
    if mode == "batch":
        return (input_tokens / 1_000_000 * BATCH_PRICE_INPUT_PER_1M) + (output_tokens / 1_000_000 * BATCH_PRICE_OUTPUT_PER_1M)
    if mode == "direct":
        return (input_tokens / 1_000_000 * DIRECT_PRICE_INPUT_PER_1M) + (output_tokens / 1_000_000 * DIRECT_PRICE_OUTPUT_PER_1M)
    raise ValueError(mode)


def build_request_body(text):
    return {
        "model": MODEL,
        "input": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Methods text:\n\n{text}\n"},
        ],
        "text": {"format": TEXT_FORMAT},
        "safety_identifier": SAFETY_IDENTIFIER,
        "store": False,
    }


def extract_output_text(body):
    if body is None:
        return None
    output_text = body.get("output_text")
    if output_text:
        return output_text
    chunks = []
    for item in body.get("output", []):
        for content_item in item.get("content", []):
            if content_item.get("type") in {"output_text", "text"}:
                text_value = content_item.get("text")
                if isinstance(text_value, dict):
                    text_value = text_value.get("value")
                if text_value:
                    chunks.append(text_value)
    return "".join(chunks) if chunks else None


In [ ]:
# =========================================================
# BUILD OR REFRESH CORPUS FILE MANIFEST
# =========================================================
def completed_label_keys_by_mode():
    keys = {
        "batch": {"global_indices": set(), "relative_paths": set()},
        "direct": {"global_indices": set(), "relative_paths": set()},
    }

    for mode, runs_dir in [("batch", BATCH_RUNS_DIR), ("direct", DIRECT_RUNS_DIR)]:
        for results_file in runs_dir.glob("*/results.csv"):
            try:
                df = pd.read_csv(results_file, usecols=lambda c: c in {"global_index", "relative_path", "status"})
            except Exception:
                continue
            if "status" in df.columns:
                df = df[df["status"].astype(str).str.lower() == "ok"]
            if "global_index" in df.columns:
                keys[mode]["global_indices"].update(pd.to_numeric(df["global_index"], errors="coerce").dropna().astype(int))
            if "relative_path" in df.columns:
                keys[mode]["relative_paths"].update(df["relative_path"].dropna().astype(str))
    return keys


def apply_label_progress_flags(manifest):
    keys = completed_label_keys_by_mode()
    manifest["labelled_by_batch"] = (
        manifest["global_index"].isin(keys["batch"]["global_indices"])
        | manifest["relative_path"].isin(keys["batch"]["relative_paths"])
    )
    manifest["labelled_by_direct"] = (
        manifest["global_index"].isin(keys["direct"]["global_indices"])
        | manifest["relative_path"].isin(keys["direct"]["relative_paths"])
    )
    manifest["already_labelled"] = manifest["labelled_by_batch"] | manifest["labelled_by_direct"]
    return manifest


def refresh_label_progress():
    global file_manifest, batch_plan
    file_manifest = build_file_manifest(force=False)
    batch_plan = build_batch_plan(force=False)
    return file_manifest, batch_plan


def completed_label_keys():
    keys = completed_label_keys_by_mode()
    done_global_indices = keys["batch"]["global_indices"] | keys["direct"]["global_indices"]
    done_relative_paths = keys["batch"]["relative_paths"] | keys["direct"]["relative_paths"]
    return done_global_indices, done_relative_paths


def completed_relative_paths():
    return completed_label_keys()[1]


def build_file_manifest(force=False):
    if FILE_MANIFEST_FILE.exists() and not force:
        manifest = pd.read_csv(FILE_MANIFEST_FILE)
    else:
        files = sorted(INPUT_FOLDER.rglob("*.txt"))
        rows = []
        for global_index, filepath in enumerate(files, start=1):
            relative_path = filepath.relative_to(INPUT_FOLDER).as_posix()
            file_size_bytes = filepath.stat().st_size
            rows.append({
                "global_index": global_index,
                "file_name": filepath.name,
                "relative_path": relative_path,
                "source_subfolder": filepath.relative_to(INPUT_FOLDER).parts[0],
                "file_path": str(filepath),
                "file_size_bytes": file_size_bytes,
                "input_tokens_estimate": int(math.ceil(file_size_bytes * INPUT_TOKEN_ESTIMATE_PER_BYTE)),
            })
        manifest = pd.DataFrame(rows)

    manifest = apply_label_progress_flags(manifest)
    manifest.to_csv(FILE_MANIFEST_FILE, index=False, encoding="utf-8-sig")
    return manifest


file_manifest = build_file_manifest(force=False)
print(f"Files in manifest: {len(file_manifest):,}")
print(f"Already labelled files detected: {int(file_manifest['already_labelled'].sum()):,}")
file_manifest.head()


In [ ]:
# =========================================================
# BUILD OR REFRESH EDITABLE BATCH ROUTING PLAN
# =========================================================
def make_source_subfolder_summary(df):
    counts = df["source_subfolder"].value_counts().sort_index().to_dict()
    return json.dumps(counts, ensure_ascii=False)


def build_batch_plan(force=False):
    if BATCH_PLAN_FILE.exists() and not force:
        existing = pd.read_csv(BATCH_PLAN_FILE)
        keep_decisions = existing.set_index("plan_batch_id")[["processing_mode", "status", "decision_notes", "run_dir"]].to_dict("index")
    else:
        keep_decisions = {}

    batches = []
    n_files = len(file_manifest)
    for start in range(0, n_files, TARGET_FILES_PER_BATCH):
        chunk = file_manifest.iloc[start:start + TARGET_FILES_PER_BATCH].copy()
        plan_batch_id = f"plan_{len(batches) + 1:04d}"
        remaining = chunk[~chunk["already_labelled"]]
        previous = keep_decisions.get(plan_batch_id, {})
        input_tokens_remaining = int(remaining["input_tokens_estimate"].sum())
        n_remaining = int(len(remaining))
        n_labelled_by_batch = int(chunk["labelled_by_batch"].sum())
        n_labelled_by_direct = int(chunk["labelled_by_direct"].sum())
        n_already_labelled = int(chunk["already_labelled"].sum())
        n_route_overlap = int((chunk["labelled_by_batch"] & chunk["labelled_by_direct"]).sum())
        inferred_mode = "batch" if n_labelled_by_batch > 0 else "direct" if n_labelled_by_direct > 0 else "pending"
        previous_mode = str(previous.get("processing_mode", "")).strip().lower()
        if previous_mode in {"", "nan", "completed"}:
            previous_mode = inferred_mode
        default_mode = inferred_mode
        default_status = "completed" if n_remaining == 0 else "in_progress" if n_already_labelled > 0 else "not_submitted"
        previous_status = str(previous.get("status", "")).strip().lower()
        if n_remaining == 0:
            previous_status = "completed"
        elif previous_status in {"", "nan", "completed_existing"}:
            previous_status = default_status
        elif previous_status == "not_submitted" and n_already_labelled > 0:
            previous_status = default_status
        batches.append({
            "plan_batch_id": plan_batch_id,
            "start_global_index": int(chunk["global_index"].min()),
            "end_global_index": int(chunk["global_index"].max()),
            "n_files": int(len(chunk)),
            "n_labelled_by_batch": n_labelled_by_batch,
            "n_labelled_by_direct": n_labelled_by_direct,
            "n_already_labelled": n_already_labelled,
            "n_route_overlap": n_route_overlap,
            "n_remaining": n_remaining,
            "source_subfolders_json": make_source_subfolder_summary(chunk),
            "input_tokens_estimate_all": int(chunk["input_tokens_estimate"].sum()),
            "input_tokens_estimate_remaining": input_tokens_remaining,
            "estimated_batch_cost_remaining_usd": round(estimate_cost(input_tokens_remaining, n_remaining, "batch"), 4),
            "estimated_direct_cost_remaining_usd": round(estimate_cost(input_tokens_remaining, n_remaining, "direct"), 4),
            "processing_mode": previous_mode or default_mode,
            "status": previous_status or default_status,
            "decision_notes": previous.get("decision_notes", ""),
            "run_dir": previous.get("run_dir", ""),
        })

    plan = pd.DataFrame(batches)
    plan.to_csv(BATCH_PLAN_FILE, index=False, encoding="utf-8-sig")
    return plan


batch_plan = build_batch_plan(force=False)
print(f"Plan batches: {len(batch_plan):,}")
print(batch_plan["processing_mode"].value_counts(dropna=False).to_string())
batch_plan.head(12)


## Decide how to process batches

Edit `processing_mode` in the CSV or in the dataframe below:

- `batch`: submit this plan batch through the OpenAI Batch API
- `direct`: process synchronously through direct API calls
- `skip`: leave it out for now
- `pending`: undecided

After editing the CSV outside the notebook, rerun the next cell to reload it and refresh progress counts from existing results.

Practical workflow:
1. Open `llm_labelled_data/routing/filtered_texts_batch_routing_plan.csv` in Excel/LibreOffice.
2. Change the `processing_mode` column for each `plan_batch_id`.
3. Save the CSV.
4. Rerun the reload cell below.
5. Use the Batch submission cell for rows marked `batch`.
6. Use the Direct processing cell only for rows marked `direct`.

Tip: leave `plan_0001` as `completed`; it corresponds to the already finished 4,500-file batch.


In [ ]:
# Reload and inspect the editable routing plan.
# 
# This cell does not submit anything. It reloads your decisions and refreshes
# progress counts from existing batch/direct results:
# llm_labelled_data/routing/filtered_texts_batch_routing_plan.csv
#
# The important editable column is processing_mode:
# - batch: submit via OpenAI Batch API
# - direct: submit via direct/synchronous API calls
# - skip: leave out for now
# - pending: undecided
#
# You can either edit the CSV externally, or uncomment/adapt the example lines below.
file_manifest, batch_plan = refresh_label_progress()

# Example edits, uncomment and adapt:
# batch_plan.loc[batch_plan["plan_batch_id"].isin(["plan_0002"]), "processing_mode"] = "batch"
# batch_plan.loc[batch_plan["plan_batch_id"].isin(["plan_0003"]), "processing_mode"] = "direct"
# batch_plan.to_csv(BATCH_PLAN_FILE, index=False, encoding="utf-8-sig")

display_cols = [
    "plan_batch_id",
    "start_global_index",
    "end_global_index",
    "n_files",
    "n_labelled_by_batch",
    "n_labelled_by_direct",
    "n_already_labelled",
    "n_route_overlap",
    "n_remaining",
    "input_tokens_estimate_remaining",
    "estimated_batch_cost_remaining_usd",
    "estimated_direct_cost_remaining_usd",
    "processing_mode",
    "status",
    "run_dir",
]
batch_plan[display_cols].head(25)


In [ ]:

batch_plan

In [ ]:
# =========================================================
# BATCH API SUBMISSION FOR ROWS MARKED processing_mode == "batch"
# =========================================================
# What this cell does:
# - Finds rows in the routing CSV where processing_mode == "batch".
# - Ignores rows already submitted/completed.
# - Builds one request JSONL + manifest per selected plan batch.
# - Submits them to the OpenAI Batch API.
# - Updates the routing CSV with status and run_dir.
#
# Safety behavior:
# - DRY_RUN = True means preview only; nothing is uploaded or submitted.
# - Set DRY_RUN = False only after the preview looks right.
# - SUBMIT_PLAN_IDS lets you submit only specific plan batches.
# - ACTIVE_TOKEN_CAP prevents overfilling the Batch queue.
#
# Examples:
#   Preview all rows marked batch:
#       DRY_RUN = True
#       SUBMIT_PLAN_IDS = []
#
#   Submit only plan_0002:
#       DRY_RUN = False
#       SUBMIT_PLAN_IDS = ["plan_0002"]
#
#   Submit plan_0002 and plan_0003 if there is queue capacity:
#       DRY_RUN = False
#       SUBMIT_PLAN_IDS = ["plan_0002", "plan_0003"]
def openai_json_request(api_key, method, url, body=None, headers=None, timeout=900):
    headers = headers or {}
    request = urllib.request.Request(
        url,
        data=body,
        headers={"Authorization": f"Bearer {api_key}", **headers},
        method=method,
    )
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))


def upload_batch_file(api_key, path):
    boundary = "----codex-boundary-" + uuid.uuid4().hex
    file_bytes = Path(path).read_bytes()
    parts = [
        f"--{boundary}\r\n".encode("utf-8"),
        b'Content-Disposition: form-data; name="purpose"\r\n\r\n',
        b"batch\r\n",
        f"--{boundary}\r\n".encode("utf-8"),
        f'Content-Disposition: form-data; name="file"; filename="{Path(path).name}"\r\n'.encode("utf-8"),
        b"Content-Type: application/jsonl\r\n\r\n",
        file_bytes,
        b"\r\n",
        f"--{boundary}--\r\n".encode("utf-8"),
    ]
    return openai_json_request(
        api_key,
        "POST",
        "https://api.openai.com/v1/files",
        b"".join(parts),
        {"Content-Type": f"multipart/form-data; boundary={boundary}"},
    )


def create_batch(api_key, input_file_id, metadata):
    body = json.dumps({
        "input_file_id": input_file_id,
        "endpoint": "/v1/responses",
        "completion_window": "24h",
        "metadata": metadata,
    }).encode("utf-8")
    return openai_json_request(api_key, "POST", "https://api.openai.com/v1/batches", body, {"Content-Type": "application/json"})


def retrieve_batch(api_key, batch_id):
    return openai_json_request(api_key, "GET", f"https://api.openai.com/v1/batches/{batch_id}")


def active_estimated_tokens(plan):
    active_statuses = {"submitted", "validating", "in_progress", "finalizing"}
    active = plan[
        (plan["processing_mode"].astype(str).str.lower() == "batch")
        & (plan["status"].astype(str).str.lower().isin(active_statuses))
    ]
    return int(active["input_tokens_estimate_remaining"].fillna(0).sum())


def files_for_plan_row(plan_row):
    start = int(plan_row["start_global_index"])
    end = int(plan_row["end_global_index"])
    subset = file_manifest[
        (file_manifest["global_index"] >= start)
        & (file_manifest["global_index"] <= end)
        & (~file_manifest["already_labelled"])
    ].copy()
    return subset


def build_batch_request_files(plan_row):
    run_name = f"batch_filtered_texts_{plan_row['plan_batch_id']}_{PROMPT_REVISION}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    run_dir = BATCH_RUNS_DIR / run_name
    run_dir.mkdir(parents=True, exist_ok=False)
    requests_file = run_dir / "requests.jsonl"
    manifest_file = run_dir / "manifest.csv"
    batch_info_file = run_dir / "batch_info.json"
    batch_output_file = run_dir / "batch_output.jsonl"
    batch_error_file = run_dir / "batch_error.jsonl"
    results_file = run_dir / "results.csv"

    subset = files_for_plan_row(plan_row)
    manifest_rows = []
    with requests_file.open("w", encoding="utf-8") as fout:
        for row in subset.to_dict("records"):
            filepath = Path(row["file_path"])
            raw_text = read_txt_file(filepath)
            normalized_text = normalize_whitespace(raw_text)
            cleaned_text, was_truncated = prepare_text(raw_text)
            custom_id = f"{int(row['global_index']):07d}__{filepath.stem}"
            manifest_rows.append({
                **row,
                "custom_id": custom_id,
                "original_text_char_len": len(normalized_text),
                "text_char_len": len(cleaned_text),
                "was_truncated": was_truncated,
            })
            request_record = {
                "custom_id": custom_id,
                "method": "POST",
                "url": "/v1/responses",
                "body": build_request_body(cleaned_text),
            }
            fout.write(json.dumps(request_record, ensure_ascii=False) + "\n")

    pd.DataFrame(manifest_rows).to_csv(manifest_file, index=False, encoding="utf-8-sig")
    return {
        "run_name": run_name,
        "run_dir": run_dir,
        "requests_file": requests_file,
        "manifest_file": manifest_file,
        "batch_info_file": batch_info_file,
        "batch_output_file": batch_output_file,
        "batch_error_file": batch_error_file,
        "results_file": results_file,
        "n_requests": len(manifest_rows),
    }



In [ ]:
remaining_batches = batch_plan[['plan_batch_id', 'processing_mode','status', 'n_remaining']][(batch_plan['status'] == 'in_progress') & (batch_plan['processing_mode'] == 'batch')]['plan_batch_id'].to_list()

print(remaining_batches)

In [ ]:
remaining_batches[:6]

In [ ]:
batch_plan[
    (batch_plan["processing_mode"].astype(str).str.lower() == "batch")
    & (~batch_plan["status"].astype(str).str.lower().isin({"submitted", "validating", "in_progress", "finalizing", "completed", "completed_existing"}))
].copy()

In [ ]:

# Safety defaults: dry-run first. Set DRY_RUN = False when ready.
DRY_RUN = False
remaining_batches = batch_plan[['plan_batch_id', 'processing_mode','status', 'n_remaining']][(batch_plan['status'] == 'in_progress') & (batch_plan['processing_mode'] == 'batch')]['plan_batch_id'].to_list()

SUBMIT_PLAN_IDS = remaining_batches[:6]  # e.g. ["plan_0002"]; empty means all rows marked processing_mode == "batch".

batch_plan = pd.read_csv(BATCH_PLAN_FILE)
candidate = batch_plan[
    (batch_plan["processing_mode"].astype(str).str.lower() == "batch")
    & (~batch_plan["status"].astype(str).str.lower().isin({"submitted", "validating", "finalizing", "completed", "completed_existing"}))
].copy()
if SUBMIT_PLAN_IDS:
    candidate = candidate[candidate["plan_batch_id"].isin(SUBMIT_PLAN_IDS)].copy()

print(f"Candidate batch rows: {len(candidate)}")
print(f"Active estimated tokens before submit: {active_estimated_tokens(batch_plan):,}")
candidate[["plan_batch_id", "n_remaining", "input_tokens_estimate_remaining", "estimated_batch_cost_remaining_usd", "processing_mode", "status"]]

if not DRY_RUN and len(candidate) > 0:
    api_key = require_api_key()
    submitted = []
    for _, plan_row in candidate.iterrows():
        active_tokens = active_estimated_tokens(batch_plan)
        planned_tokens = int(plan_row["input_tokens_estimate_remaining"])
        if active_tokens + planned_tokens > ACTIVE_TOKEN_CAP:
            print(f"Stopping before {plan_row['plan_batch_id']}: active+planned tokens would exceed cap.")
            break

        files = build_batch_request_files(plan_row)
        uploaded = upload_batch_file(api_key, files["requests_file"])
        batch = create_batch(api_key, uploaded["id"], {
            "run_name": files["run_name"][:512],
            "plan_batch_id": str(plan_row["plan_batch_id"]),
            "model": MODEL,
            "prompt_revision": PROMPT_REVISION,
            "n_requests": str(files["n_requests"]),
        })
        info = {
            "run_name": files["run_name"],
            "plan_batch_id": plan_row["plan_batch_id"],
            "model": MODEL,
            "prompt_variant": PROMPT_VARIANT,
            "prompt_revision": PROMPT_REVISION,
            "input_file_id": uploaded.get("id"),
            "batch_id": batch.get("id"),
            "status": batch.get("status"),
            "request_file": str(files["requests_file"]),
            "manifest_file": str(files["manifest_file"]),
            "batch_output_file": str(files["batch_output_file"]),
            "batch_error_file": str(files["batch_error_file"]),
            "results_file": str(files["results_file"]),
            "n_requests": files["n_requests"],
            "input_tokens_estimate": int(plan_row["input_tokens_estimate_remaining"]),
            "estimated_cost_usd": float(plan_row["estimated_batch_cost_remaining_usd"]),
            "submitted_at_utc": datetime.now(timezone.utc).isoformat(),
            "batch_api_raw_status": batch,
        }
        files["batch_info_file"].write_text(json.dumps(info, indent=2, default=str), encoding="utf-8")

        mask = batch_plan["plan_batch_id"] == plan_row["plan_batch_id"]
        batch_plan.loc[mask, "status"] = batch.get("status") or "submitted"
        batch_plan.loc[mask, "run_dir"] = str(files["run_dir"])
        batch_plan.to_csv(BATCH_PLAN_FILE, index=False, encoding="utf-8-sig")
        submitted.append(info)
        print(f"Submitted {plan_row['plan_batch_id']}: batch_id={batch.get('id')} status={batch.get('status')}")

    submitted


In [ ]:
# =========================================================
# CHECK/DOWNLOAD/PARSE SUBMITTED BATCH RUNS
# =========================================================
# Run this cell whenever you want to check submitted Batch jobs.
#
# It will:
# - Query OpenAI for each submitted batch linked to this routing plan.
# - Download output/error files if they are available.
# - Parse completed batch outputs into results.csv.
# - Update batch_info.json and the routing CSV status.
#
# You can use parsed results from one completed batch immediately, even if other
# batches are still validating/in_progress/finalizing.
import csv

CSV_RESULT_WRITE_OPTIONS = {
    "index": False,
    "encoding": "utf-8-sig",
    "quoting": csv.QUOTE_MINIMAL,
    "doublequote": True,
    "escapechar": "\\",
    "lineterminator": "\n",
}


def write_results_csv(frame, target_path):
    frame.to_csv(target_path, **CSV_RESULT_WRITE_OPTIONS)


def download_file(api_key, file_id, target_path):
    req = urllib.request.Request(f"https://api.openai.com/v1/files/{file_id}/content", headers={"Authorization": f"Bearer {api_key}"})
    with urllib.request.urlopen(req, timeout=600) as resp:
        Path(target_path).write_bytes(resp.read())


def parse_batch_run(batch_info_file):
    info = json.loads(Path(batch_info_file).read_text(encoding="utf-8"))
    manifest = pd.read_csv(info["manifest_file"])
    manifest_by_id = {row["custom_id"]: row for row in manifest.to_dict("records")}
    output_file = Path(info["batch_output_file"])
    error_file = Path(info["batch_error_file"])
    rows = []
    handled = set()

    def base_row(base):
        return {
            "custom_id": base.get("custom_id"),
            "global_index": base.get("global_index"),
            "file_name": base.get("file_name"),
            "relative_path": base.get("relative_path"),
            "source_subfolder": base.get("source_subfolder"),
            "file_path": base.get("file_path"),
            "original_text_char_len": base.get("original_text_char_len"),
            "text_char_len": base.get("text_char_len"),
            "was_truncated": base.get("was_truncated"),
            "input_tokens_estimate": base.get("input_tokens_estimate"),
            "prompt_variant": info.get("prompt_variant"),
            "prompt_revision": info.get("prompt_revision"),
        }

    if output_file.exists():
        with output_file.open("r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                record = json.loads(line)
                custom_id = record.get("custom_id")
                handled.add(custom_id)
                response = record.get("response") or {}
                body = response.get("body") or {}
                usage = body.get("usage") or {}
                output_text = extract_output_text(body)
                row = base_row(manifest_by_id.get(custom_id, {}))
                try:
                    parsed = json.loads(output_text)
                    data_sources = deduplicate_sources(parsed.get("data_sources", []))
                    db_use = int(parsed.get("db_use", 0))
                    if data_sources:
                        db_use = 1
                    row.update({
                        "db_use": db_use,
                        "data_sources_json": serialize_json_field(data_sources),
                        "data_resources_json": serialize_json_field(data_sources),
                        "n_data_sources": len(data_sources),
                        "reason_short": parsed.get("reason_short", ""),
                        "input_tokens": usage.get("input_tokens"),
                        "output_tokens": usage.get("output_tokens"),
                        "total_tokens": usage.get("total_tokens"),
                        "response_status_code": response.get("status_code"),
                        "status": "ok",
                        "error_message": None,
                        "raw_output_text": output_text,
                    })
                except Exception as exc:
                    row.update({
                        "db_use": None,
                        "data_sources_json": None,
                        "data_resources_json": None,
                        "n_data_sources": None,
                        "reason_short": None,
                        "input_tokens": usage.get("input_tokens"),
                        "output_tokens": usage.get("output_tokens"),
                        "total_tokens": usage.get("total_tokens"),
                        "response_status_code": response.get("status_code"),
                        "status": "parse_error",
                        "error_message": str(exc),
                        "raw_output_text": output_text,
                    })
                rows.append(row)

    if error_file.exists():
        with error_file.open("r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                record = json.loads(line)
                custom_id = record.get("custom_id")
                handled.add(custom_id)
                error = record.get("error") or {}
                row = base_row(manifest_by_id.get(custom_id, {}))
                row.update({
                    "db_use": None,
                    "data_sources_json": None,
                    "data_resources_json": None,
                    "n_data_sources": None,
                    "reason_short": None,
                    "input_tokens": None,
                    "output_tokens": None,
                    "total_tokens": None,
                    "response_status_code": None,
                    "status": f"batch_error: {error.get('code', 'unknown')}",
                    "error_message": error.get("message", json.dumps(error)),
                    "raw_output_text": None,
                })
                rows.append(row)

    for custom_id, base in manifest_by_id.items():
        if custom_id in handled:
            continue
        row = base_row(base)
        row.update({
            "db_use": None,
            "data_sources_json": None,
            "data_resources_json": None,
            "n_data_sources": None,
            "reason_short": None,
            "input_tokens": None,
            "output_tokens": None,
            "total_tokens": None,
            "response_status_code": None,
            "status": "missing_from_batch_output",
            "error_message": "No success or error line found for this custom_id.",
            "raw_output_text": None,
        })
        rows.append(row)

    results = pd.DataFrame(rows).sort_values("custom_id").reset_index(drop=True)
    write_results_csv(results, info["results_file"])
    return info, results


api_key = require_api_key()
batch_plan = pd.read_csv(BATCH_PLAN_FILE)
checked = []

for batch_info_file in sorted(BATCH_RUNS_DIR.glob("*/batch_info.json")):
    info = json.loads(batch_info_file.read_text(encoding="utf-8"))
    if info.get("prompt_revision") != PROMPT_REVISION:
        continue
    if not info.get("plan_batch_id"):
        continue

    batch = retrieve_batch(api_key, info["batch_id"])
    info.update({
        "status": batch.get("status"),
        "output_file_id": batch.get("output_file_id"),
        "error_file_id": batch.get("error_file_id"),
        "request_counts": batch.get("request_counts"),
        "completed_at": batch.get("completed_at"),
        "failed_at": batch.get("failed_at"),
        "expired_at": batch.get("expired_at"),
        "last_checked_at_utc": datetime.now(timezone.utc).isoformat(),
        "batch_api_raw_status": batch,
    })

    if batch.get("output_file_id") and not Path(info["batch_output_file"]).exists():
        download_file(api_key, batch["output_file_id"], info["batch_output_file"])
    if batch.get("error_file_id") and not Path(info["batch_error_file"]).exists():
        download_file(api_key, batch["error_file_id"], info["batch_error_file"])

    batch_info_file.write_text(json.dumps(info, indent=2, default=str), encoding="utf-8")

    if Path(info["batch_output_file"]).exists() or Path(info["batch_error_file"]).exists():
        info, results = parse_batch_run(batch_info_file)
        ok = int((results["status"] == "ok").sum())
        err = int((results["status"] != "ok").sum())
    else:
        ok = 0
        err = 0

    mask = batch_plan["plan_batch_id"] == info["plan_batch_id"]
    if mask.any():
        batch_plan.loc[mask, "status"] = info.get("status")
        batch_plan.loc[mask, "run_dir"] = str(Path(batch_info_file).parent)

    checked.append({
        "plan_batch_id": info.get("plan_batch_id"),
        "batch_id": info.get("batch_id"),
        "status": info.get("status"),
        "request_counts": info.get("request_counts"),
        "ok_rows_parsed": ok,
        "error_rows_parsed": err,
        "run_dir": str(Path(batch_info_file).parent),
    })

batch_plan.to_csv(BATCH_PLAN_FILE, index=False, encoding="utf-8-sig")
file_manifest, batch_plan = refresh_label_progress()
pd.DataFrame(checked)


## direct job

In [ ]:
# =========================================================
# DIRECT PROCESSING FOR ROWS MARKED processing_mode == "direct"
# =========================================================
# Direct processing is synchronous: each request runs immediately and the results
# are appended to a local results.csv as they come back.
#
# Use direct mode for small or urgent subsets only. It is more expensive than Batch
# and consumes normal realtime API rate limits, but it can run while a Batch job is
# still processing.
#
# Safety behavior:
# - RUN_DIRECT = False means this cell only prints a reminder.
# - DIRECT_PLAN_IDS controls which plan batches marked direct are processed.
# - MAX_DIRECT_FILES_THIS_RUN limits how many files are sent in one notebook run.
# - Before sending anything, the cell refreshes already-labelled files from all
#   existing batch/direct results and skips successful rows automatically.
#
# Example:
#   RUN_DIRECT = True
#   DIRECT_PLAN_IDS = ["plan_0005"]
#   MAX_DIRECT_FILES_THIS_RUN = 25
def call_responses_api(api_key, text):
    body = json.dumps(build_request_body(text)).encode("utf-8")
    req = urllib.request.Request(
        "https://api.openai.com/v1/responses",
        data=body,
        headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=180) as resp:
        return json.loads(resp.read().decode("utf-8"))


def update_direct_progress_in_batch_plan(run_dir, initial_plan_state, touched_plan_ids, processed_ok_by_plan, processed_input_estimate_by_plan, final=False):
    plan = pd.read_csv(BATCH_PLAN_FILE)
    for plan_batch_id in sorted(touched_plan_ids):
        if plan_batch_id not in initial_plan_state.index:
            continue
        mask = plan["plan_batch_id"] == plan_batch_id
        if not mask.any():
            continue

        initial = initial_plan_state.loc[plan_batch_id]
        ok_added = int(processed_ok_by_plan.get(plan_batch_id, 0))
        input_tokens_done = int(processed_input_estimate_by_plan.get(plan_batch_id, 0))
        n_files = int(initial["n_files"])
        n_already = min(n_files, int(initial["n_already_labelled"]) + ok_added)
        n_direct = min(n_files, int(initial["n_labelled_by_direct"]) + ok_added)
        n_remaining = max(0, n_files - n_already)
        input_tokens_remaining = max(0, int(initial["input_tokens_estimate_remaining"]) - input_tokens_done)

        plan.loc[mask, "n_labelled_by_direct"] = n_direct
        plan.loc[mask, "n_already_labelled"] = n_already
        if "n_route_overlap" in plan.columns:
            plan.loc[mask, "n_route_overlap"] = int(initial.get("n_route_overlap", 0))
        plan.loc[mask, "n_remaining"] = n_remaining
        plan.loc[mask, "input_tokens_estimate_remaining"] = input_tokens_remaining
        plan.loc[mask, "estimated_batch_cost_remaining_usd"] = round(estimate_cost(input_tokens_remaining, n_remaining, "batch"), 4)
        plan.loc[mask, "estimated_direct_cost_remaining_usd"] = round(estimate_cost(input_tokens_remaining, n_remaining, "direct"), 4)
        plan.loc[mask, "run_dir"] = str(run_dir)

        if n_remaining == 0:
            plan.loc[mask, "status"] = "completed"
        elif final and ok_added > 0:
            plan.loc[mask, "status"] = "in_progress"
        elif final:
            plan.loc[mask, "status"] = "errors_or_no_successes"
        else:
            plan.loc[mask, "status"] = "in_progress"

    plan.to_csv(BATCH_PLAN_FILE, index=False, encoding="utf-8-sig")
    return plan


In [ ]:
batch_plan[['plan_batch_id', 'processing_mode','status', 'n_remaining']]

In [ ]:
RUN_DIRECT = True
DIRECT_PLAN_IDS = ["plan_0053"]  # e.g. ["plan_0003"]
MAX_DIRECT_FILES_THIS_RUN = 1000
PLAN_PROGRESS_UPDATE_EVERY = 50

if RUN_DIRECT:
    api_key = require_api_key()

    # Refresh first, so reruns continue from the first unlabelled file in each
    # selected plan batch instead of repeating rows from earlier direct runs.
    file_manifest, batch_plan = refresh_label_progress()
    selected_plan = batch_plan[batch_plan["processing_mode"].astype(str).str.lower() == "direct"].copy()
    if DIRECT_PLAN_IDS:
        selected_plan = selected_plan[selected_plan["plan_batch_id"].isin(DIRECT_PLAN_IDS)].copy()
    initial_plan_state = batch_plan.set_index("plan_batch_id").copy()

    files_to_process = []
    planned_counts = []
    for _, plan_row in selected_plan.iterrows():
        subset = files_for_plan_row(plan_row)
        planned_counts.append({
            "plan_batch_id": plan_row["plan_batch_id"],
            "remaining_before_run": int(len(subset)),
        })
        subset = subset.assign(plan_batch_id=plan_row["plan_batch_id"])
        files_to_process.extend(subset.to_dict("records"))
    files_to_process = files_to_process[:MAX_DIRECT_FILES_THIS_RUN]

    if not files_to_process:
        print("No unlabelled files found for the selected direct plan batches.")
        pd.DataFrame(planned_counts)
    else:
        run_name = f"direct_filtered_texts_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        run_dir = DIRECT_RUNS_DIR / run_name
        run_dir.mkdir(parents=True, exist_ok=False)
        results_file = run_dir / "results.csv"
        rows = []
        touched_plan_ids = {row["plan_batch_id"] for row in files_to_process}
        processed_ok_by_plan = {}
        processed_input_estimate_by_plan = {}
        print(f"Direct run will process {len(files_to_process):,} unlabelled files.")
        display(pd.DataFrame(planned_counts))
        batch_plan = update_direct_progress_in_batch_plan(
            run_dir,
            initial_plan_state,
            touched_plan_ids,
            processed_ok_by_plan,
            processed_input_estimate_by_plan,
            final=False,
        )

        for row in files_to_process:
            filepath = Path(row["file_path"])
            raw_text = read_txt_file(filepath)
            normalized_text = normalize_whitespace(raw_text)
            cleaned_text, was_truncated = prepare_text(raw_text)
            try:
                response = call_responses_api(api_key, cleaned_text)
                output_text = extract_output_text(response)
                parsed = json.loads(output_text)
                data_sources = deduplicate_sources(parsed.get("data_sources", []))
                db_use = int(parsed.get("db_use", 0))
                if data_sources:
                    db_use = 1
                usage = response.get("usage") or {}
                status = "ok"
                error_message = None
            except Exception as exc:
                parsed = {}
                data_sources = []
                db_use = None
                usage = {}
                output_text = None
                status = "error"
                error_message = str(exc)

            rows.append({
                **row,
                "original_text_char_len": len(normalized_text),
                "text_char_len": len(cleaned_text),
                "was_truncated": was_truncated,
                "prompt_variant": PROMPT_VARIANT,
                "prompt_revision": PROMPT_REVISION,
                "db_use": db_use,
                "data_sources_json": serialize_json_field(data_sources),
                "data_resources_json": serialize_json_field(data_sources),
                "n_data_sources": len(data_sources),
                "reason_short": parsed.get("reason_short", ""),
                "input_tokens": usage.get("input_tokens"),
                "output_tokens": usage.get("output_tokens"),
                "total_tokens": usage.get("total_tokens"),
                "status": status,
                "error_message": error_message,
                "raw_output_text": output_text,
            })
            pd.DataFrame(rows).to_csv(
                results_file,
                index=False,
                encoding="utf-8-sig",
                quoting=csv.QUOTE_MINIMAL,
                doublequote=True,
                escapechar="\\",
                lineterminator="\n",
            )

            if status == "ok":
                plan_batch_id = row["plan_batch_id"]
                processed_ok_by_plan[plan_batch_id] = processed_ok_by_plan.get(plan_batch_id, 0) + 1
                processed_input_estimate_by_plan[plan_batch_id] = (
                    processed_input_estimate_by_plan.get(plan_batch_id, 0)
                    + int(row.get("input_tokens_estimate", 0) or 0)
                )
            if len(rows) % PLAN_PROGRESS_UPDATE_EVERY == 0:
                batch_plan = update_direct_progress_in_batch_plan(
                    run_dir,
                    initial_plan_state,
                    touched_plan_ids,
                    processed_ok_by_plan,
                    processed_input_estimate_by_plan,
                    final=False,
                )

        batch_plan = update_direct_progress_in_batch_plan(
            run_dir,
            initial_plan_state,
            touched_plan_ids,
            processed_ok_by_plan,
            processed_input_estimate_by_plan,
            final=True,
        )

        file_manifest, batch_plan = refresh_label_progress()
        for plan_batch_id in touched_plan_ids:
            mask = batch_plan["plan_batch_id"] == plan_batch_id
            if not mask.any():
                continue
            remaining = int(batch_plan.loc[mask, "n_remaining"].iloc[0])
            batch_plan.loc[mask, "status"] = "completed" if remaining == 0 else "in_progress"
            batch_plan.loc[mask, "run_dir"] = str(run_dir)
        batch_plan.to_csv(BATCH_PLAN_FILE, index=False, encoding="utf-8-sig")

        print(f"Direct results saved to: {results_file}")
        pd.DataFrame(rows).head()
else:
    print("RUN_DIRECT is False. Set it to True only for small direct-processing subsets.")
